# 01_silver_time_series_clean

## Purpose
Transform Bronze time series tables to Silver layer with:
- Column name standardization (snake_case)
- Null handling
- Deduplication
- Date parsing
- Data quality validation
- Quarantine routing for failed records

## Source Tables (Bronze)
- `zillow.zillow_bronze.zip_ts_bronze`
- `zillow.zillow_bronze.city_ts_bronze`
- `zillow.zillow_bronze.county_ts_json_bronze`
- `zillow.zillow_bronze.state_ts_xml_bronze`
- `zillow.zillow_bronze.metro_ts_bronze`
- `zillow.zillow_bronze.neighborhood_ts_dlt_bronze`

## Target Tables (Silver)
- `zillow.zillow_silver.*_ts_silver`

In [0]:
# COMMAND ----------
# Import UDF library
%run "./common/udf_library"

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
import uuid

# Configuration
CAT = "zillow"
BRONZE = "zillow_bronze"
SILVER = "zillow_silver"
QUARANTINE = "zillow_quarantine"

run_id = str(uuid.uuid4())

# Create Silver and Quarantine schemas if not exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CAT}.{SILVER}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CAT}.{QUARANTINE}")

print(f"Run ID: {run_id}")

In [0]:
def transform_time_series_to_silver(
    source_table: str,
    target_table: str,
    date_col: str = "Date",
    region_col: str = "RegionName"
):
    """
    Transform a Bronze time series table to Silver.
    
    Steps:
    1. Read Bronze table
    2. Standardize column names
    3. Parse date column
    4. Handle nulls
    5. Deduplicate
    6. Add audit columns
    7. Validate and route to quarantine if needed
    8. Write to Silver
    
    Args:
        source_table: Full Bronze table name
        target_table: Full Silver table name
        date_col: Name of date column in source
        region_col: Name of region column in source
    """
    print(f"\n{'='*60}")
    print(f"Transforming: {source_table} -> {target_table}")
    print(f"{'='*60}")
    
    # 1. Read Bronze
    try:
        df = spark.table(source_table)
        bronze_count = df.count()
        print(f"1. Read {bronze_count:,} rows from Bronze")
    except Exception as e:
        print(f"ERROR: Could not read {source_table}: {e}")
        return
    
    # Drop Bronze audit columns (we'll add Silver ones)
    bronze_audit_cols = ['load_dt', 'source_path', 'source_file', 'ingest_mode', 'ingest_run_id', 'part_id']
    for col in bronze_audit_cols:
        if col in df.columns:
            df = df.drop(col)
    
    # 2. Standardize column names
    df = standardize_dataframe_columns(df)
    print(f"2. Standardized {len(df.columns)} column names to snake_case")
    
    # Find the standardized date and region columns
    date_col_std = standardize_column_name(date_col)
    region_col_std = standardize_column_name(region_col)
    
    # 3. Parse date column
    if date_col_std in df.columns:
        df = df.withColumn(
            date_col_std, 
            F.to_date(F.col(date_col_std).cast("string"), "yyyy-MM-dd")
        )
        print(f"3. Parsed '{date_col_std}' to DATE type")
    
    # 4. Handle nulls for numeric columns - replace with null (keep as is)
    # We'll keep nulls but ensure they're proper nulls, not empty strings
    for col_name in df.columns:
        if col_name not in [date_col_std, region_col_std]:
            df = df.withColumn(
                col_name,
                F.when(F.col(col_name) == "", None).otherwise(F.col(col_name))
            )
    print("4. Cleaned empty strings to nulls")
    
    # 5. Deduplicate - keep first occurrence per date/region
    if date_col_std in df.columns and region_col_std in df.columns:
        before_dedup = df.count()
        df = df.dropDuplicates([date_col_std, region_col_std])
        after_dedup = df.count()
        print(f"5. Deduplicated: {before_dedup:,} -> {after_dedup:,} rows (removed {before_dedup - after_dedup:,} duplicates)")
    else:
        df = df.dropDuplicates()
        print("5. Deduplicated on all columns")
    
    # 6. Add audit columns
    df = (df
        .withColumn("processed_dt", F.current_timestamp())
        .withColumn("source_table", F.lit(source_table))
        .withColumn("silver_run_id", F.lit(run_id))
        .withColumn("record_hash", F.md5(F.concat_ws("|", *[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in df.columns])))
    )
    print("6. Added audit columns (processed_dt, source_table, silver_run_id, record_hash)")
    
    # 7. Data quality validation - identify bad records
    # Required fields check
    if date_col_std in df.columns and region_col_std in df.columns:
        valid_df = df.filter(
            F.col(date_col_std).isNotNull() & 
            F.col(region_col_std).isNotNull() &
            (F.col(region_col_std) != "")
        )
        
        invalid_df = df.filter(
            F.col(date_col_std).isNull() | 
            F.col(region_col_std).isNull() |
            (F.col(region_col_std) == "")
        )
        
        valid_count = valid_df.count()
        invalid_count = invalid_df.count()
        print(f"7. Validation: {valid_count:,} valid, {invalid_count:,} invalid")
        
        # Route invalid to quarantine
        if invalid_count > 0:
            quarantine_table = f"{CAT}.{QUARANTINE}.failed_records"
            quarantine_df = (invalid_df
                .withColumn("failure_reason", F.lit("Missing required field: date or region_name"))
                .withColumn("failed_dt", F.current_timestamp())
                .select(
                    F.lit(source_table).alias("source_table"),
                    F.to_json(F.struct(*[F.col(c) for c in invalid_df.columns])).alias("record_json"),
                    "failure_reason",
                    "failed_dt"
                )
            )
            quarantine_df.write.format("delta").mode("append").saveAsTable(quarantine_table)
            print(f"   Quarantined {invalid_count:,} records to {quarantine_table}")
        
        df = valid_df
    else:
        print("7. Skipped validation (date/region columns not found)")
    
    # 8. Write to Silver
    final_count = df.count()
    df.write.format("delta").mode("overwrite").saveAsTable(target_table)
    
    spark.sql(f"COMMENT ON TABLE {target_table} IS 'Silver layer: cleaned, deduplicated, standardized from {source_table}'")
    
    print(f"8. Wrote {final_count:,} rows to Silver table: {target_table}")
    
    return {
        "source": source_table,
        "target": target_table,
        "bronze_count": bronze_count,
        "silver_count": final_count
    }

In [0]:
# Define transformations mapping
transformations = [
    # (source_bronze_table, target_silver_table)
    (f"{CAT}.{BRONZE}.zip_ts_bronze", f"{CAT}.{SILVER}.zip_ts_silver"),
    (f"{CAT}.{BRONZE}.city_ts_bronze", f"{CAT}.{SILVER}.city_ts_silver"),
    (f"{CAT}.{BRONZE}.metro_ts_bronze", f"{CAT}.{SILVER}.metro_ts_silver"),
    (f"{CAT}.{BRONZE}.county_ts_json_bronze", f"{CAT}.{SILVER}.county_ts_silver"),
]

results = []

for source, target in transformations:
    try:
        result = transform_time_series_to_silver(source, target)
        if result:
            results.append(result)
    except Exception as e:
        print(f"ERROR processing {source}: {e}")
        results.append({"source": source, "target": target, "error": str(e)})

In [0]:
# Summary
print("\n" + "="*60)
print("TRANSFORMATION SUMMARY")
print("="*60)

for r in results:
    if "error" in r:
        print(f"❌ {r['source']} -> ERROR: {r['error']}")
    else:
        print(f"✅ {r['source']} -> {r['target']}")
        print(f"   Bronze: {r['bronze_count']:,} | Silver: {r['silver_count']:,}")

# Show Silver tables
print("\n" + "="*60)
print("SILVER TABLES CREATED")
print("="*60)
display(spark.sql(f"SHOW TABLES IN {CAT}.{SILVER}"))

In [0]:
# UNIT TESTS
print("\n" + "="*60)
print("UNIT TESTS")
print("="*60)

# Test 1: Column names are snake_case
test_table = f"{CAT}.{SILVER}.zip_ts_silver"
try:
    cols = spark.table(test_table).columns
    all_snake_case = all(col == col.lower() and " " not in col for col in cols)
    print(f"Test 1 - Column names snake_case: {'PASS ✅' if all_snake_case else 'FAIL ❌'}")
except:
    print("Test 1 - SKIPPED (table not found)")

# Test 2: No duplicates on date/region
try:
    dup_count = spark.sql(f"""
        SELECT date, region_name, COUNT(*) as cnt 
        FROM {test_table} 
        GROUP BY date, region_name 
        HAVING COUNT(*) > 1
    """).count()
    print(f"Test 2 - No duplicates: {'PASS ✅' if dup_count == 0 else 'FAIL ❌'} ({dup_count} duplicate groups)")
except:
    print("Test 2 - SKIPPED (table not found)")

# Test 3: Date column is DATE type
try:
    date_type = spark.sql(f"SELECT typeof(date) as dt FROM {test_table} LIMIT 1").collect()[0]["dt"]
    print(f"Test 3 - Date is DATE type: {'PASS ✅' if date_type == 'date' else 'FAIL ❌'} (actual: {date_type})")
except:
    print("Test 3 - SKIPPED (table not found)")

# Test 4: Audit columns exist
try:
    cols = spark.table(test_table).columns
    audit_cols = ['processed_dt', 'source_table', 'silver_run_id', 'record_hash']
    has_audit = all(c in cols for c in audit_cols)
    print(f"Test 4 - Audit columns exist: {'PASS ✅' if has_audit else 'FAIL ❌'}")
except:
    print("Test 4 - SKIPPED (table not found)")